# Validador de geometrías de proveedores — EUDR
### Demo — Producto 1 del portfolio EUDR

Este notebook corre el validador contra un archivo de proveedores **sintético y sucio a propósito**
(ambientado en zonas forestales reales de Corrientes y Misiones, Argentina, con nombres y coordenadas
ficticias) para mostrar el flujo completo: **ingesta → validación → GeoJSON conforme → informe de calidad**.

Ver `README.md` para el detalle normativo de cada chequeo.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from main import run
import pandas as pd
pd.set_option("display.max_colwidth", 100)

## 1. Correr el validador sobre la planilla Excel de proveedores

Formato "largo": una fila por vértice. 15 parcelas, algunas válidas, otras con errores
sembrados a propósito (falta de coordenadas, país inconsistente, superposición entre
parcelas, etc.).

In [2]:
result = run("../data/synthetic_planilla_proveedores.xlsx", "../outputs/demo_excel")

print(f"Parcelas en el GeoJSON de salida: {result['n_output_features']}")
print(f"Parcelas excluidas por error bloqueante: {result['n_excluded_parcels']}")
print(f"Hallazgos totales: {len(result['findings'])}")

Parcelas en el GeoJSON de salida: 7
Parcelas excluidas por error bloqueante: 8
Hallazgos totales: 19


## 2. Ver los hallazgos como tabla

In [3]:
rows = [
    {"Severidad": f.severity, "ParcelID": f.parcel_id, "Código": f.code,
     "Mensaje": f.message, "Fundamento": f.reference}
    for f in sorted(result["findings"], key=lambda f: f.sort_key())
]
df_findings = pd.DataFrame(rows)
df_findings

,Severidad,ParcelID,Código,Mensaje,Fundamento
0,ERROR,MSNS-011-INVALIDISO,COUNTRY_INVALID_ISO,'ZZ' no es un código ISO 3166-1 alpha-2 válido.,EUDR GeoJSON File Description v1.5
1,ERROR,MSNS-006-LATLONSWAP,COUNTRY_MISMATCH,ProducerCountry='AR' no coincide con la ubicación real de la geometría (el centroide de la parce...,EUDR GeoJSON File Description v1.5
2,ERROR,MSNS-014-SELFINTERSECT,GEOM_SELF_INTERSECTION,"La secuencia de vértices produce una geometría inválida (auto-intersectada, tipo 'ocho'). Revisa...",EUDR GeoJSON File Description v1.5
3,ERROR,MSNS-003-MISSINGLATLON,ING_MISSING_LATLON,Falta Latitude o Longitude.,Producto 1 — esquema de entrada tabular
4,ERROR,MSNS-004-TWOVERTICES,ING_TWO_VERTICES,La parcela tiene exactamente 2 filas de vértices: no alcanza para un polígono (mínimo 3 vértices...,Producto 1 — esquema de entrada tabular
5,ERROR,MSNS-010-OVERLAP-A / MSNS-010-OVERLAP-B,PARCELS_OVERLAP,La parcela 'MSNS-010-OVERLAP-A' se superpone con 'MSNS-010-OVERLAP-B' en 7.170 ha.,EUDR GeoJSON File Description v1.5
6,ERROR,MSNS-008-4HAVIOLATION,RULE_4HA_VIOLATION,Area declarada = 8.0 ha (> 4 ha) pero la geometría es un Point. Parcelas de más de 4 ha deben de...,"Reglamento (UE) 2023/1115, Art. 9"
7,WARNING,MSNS-013-AREAMISMATCH,AREA_MISMATCH,El área calculada de la geometría (14.94 ha) supera 4 ha pero el Area declarada (1.5 ha) no. Rev...,EUDR GeoJSON File Description v1.5
8,WARNING,MSNS-007-PRECISION,COORD_PRECISION,La longitud tiene 14 decimales (máximo recomendado: 6). Más de 6 decimales puede generar coorden...,EUDR GeoJSON File Description v1.5
9,WARNING,MSNS-007-PRECISION,COORD_PRECISION,La latitud tiene 14 decimales (máximo recomendado: 6). Más de 6 decimales puede generar coordena...,EUDR GeoJSON File Description v1.5


## 3. Filtrar solo los errores bloqueantes

Estas son las parcelas que **no** entraron al `output.geojson` — quedaron documentadas acá para que
el proveedor las corrija.

In [4]:
df_findings[df_findings["Severidad"] == "ERROR"]

,Severidad,ParcelID,Código,Mensaje,Fundamento
0,ERROR,MSNS-011-INVALIDISO,COUNTRY_INVALID_ISO,'ZZ' no es un código ISO 3166-1 alpha-2 válido.,EUDR GeoJSON File Description v1.5
1,ERROR,MSNS-006-LATLONSWAP,COUNTRY_MISMATCH,ProducerCountry='AR' no coincide con la ubicación real de la geometría (el centroide de la parce...,EUDR GeoJSON File Description v1.5
2,ERROR,MSNS-014-SELFINTERSECT,GEOM_SELF_INTERSECTION,"La secuencia de vértices produce una geometría inválida (auto-intersectada, tipo 'ocho'). Revisa...",EUDR GeoJSON File Description v1.5
3,ERROR,MSNS-003-MISSINGLATLON,ING_MISSING_LATLON,Falta Latitude o Longitude.,Producto 1 — esquema de entrada tabular
4,ERROR,MSNS-004-TWOVERTICES,ING_TWO_VERTICES,La parcela tiene exactamente 2 filas de vértices: no alcanza para un polígono (mínimo 3 vértices...,Producto 1 — esquema de entrada tabular
5,ERROR,MSNS-010-OVERLAP-A / MSNS-010-OVERLAP-B,PARCELS_OVERLAP,La parcela 'MSNS-010-OVERLAP-A' se superpone con 'MSNS-010-OVERLAP-B' en 7.170 ha.,EUDR GeoJSON File Description v1.5
6,ERROR,MSNS-008-4HAVIOLATION,RULE_4HA_VIOLATION,Area declarada = 8.0 ha (> 4 ha) pero la geometría es un Point. Parcelas de más de 4 ha deben de...,"Reglamento (UE) 2023/1115, Art. 9"


## 4. Inspeccionar el GeoJSON de salida (conforme a v1.5)

In [5]:
import json

with open("../outputs/demo_excel/output.geojson", encoding="utf-8") as f:
    out = json.load(f)

print(f"type: {out['type']}")
print(f"features: {len(out['features'])}")
print()
print(json.dumps(out["features"][0], indent=2, ensure_ascii=False))

type: FeatureCollection
features: 7

{
  "type": "Feature",
  "properties": {
    "ProducerName": "Yerbatera San Ignacio",
    "ProducerCountry": "AR",
    "ProductionPlace": "Chacra 3",
    "Area": 1.8,
    "CommonName": "Pino",
    "ScientificName": "Pinus taeda"
  },
  "geometry": {
    "type": "Point",
    "coordinates": [
      -54.59,
      -26.4
    ]
  }
}


## 5. Ver el informe de calidad (versión Markdown)

Mismo contenido está disponible en `informe_calidad.pdf` y `informe_calidad.xlsx` dentro de la
carpeta de salida — pensados para un equipo de compliance que trabaja en Excel/SAP, no en notebooks.

In [6]:
from IPython.display import Markdown, display

with open("../outputs/demo_excel/informe_calidad.md", encoding="utf-8") as f:
    display(Markdown(f.read()))

# Informe de calidad — Validador de geometrías de proveedores (EUDR)

**Archivo de entrada:** `synthetic_planilla_proveedores.xlsx`  
**Fecha de generación:** 2026-09-10 16:39  
**Commodity:** Madera (Cono Sur)  
**Formato de referencia:** EUDR GeoJSON File Description v1.5 (Reglamento de Ejecución (UE) 2024/3084)

## Resumen ejecutivo

| Métrica | Valor |
|---|---|
| Parcelas evaluadas | 15 |
| Parcelas aptas para presentar tal cual | 7 |
| Parcelas con error bloqueante | 8 |
| Errores bloqueantes (total) | 7 |
| Advertencias (no bloqueantes) | 6 |
| Notas de conversión | 6 |

**Qué significa "apto":** la parcela no tiene ningún hallazgo de severidad *Error bloqueante*. El GeoJSON de salida (`output.geojson`) incluye únicamente las parcelas aptas. Las parcelas con error bloqueante quedan fuera del archivo de salida y deben corregirse en origen antes de volver a intentar la presentación.

## 🔴 Error bloqueantes (7)

- **[COUNTRY_INVALID_ISO]** — Parcela `MSNS-011-INVALIDISO`: 'ZZ' no es un código ISO 3166-1 alpha-2 válido.
  *Fundamento: EUDR GeoJSON File Description v1.5*
- **[COUNTRY_MISMATCH]** — Parcela `MSNS-006-LATLONSWAP`: ProducerCountry='AR' no coincide con la ubicación real de la geometría (el centroide de la parcela cae fuera del país declarado, incluso con margen de tolerancia).
  *Fundamento: EUDR GeoJSON File Description v1.5*
- **[GEOM_SELF_INTERSECTION]** — Parcela `MSNS-014-SELFINTERSECT`: La secuencia de vértices produce una geometría inválida (auto-intersectada, tipo 'ocho'). Revisar el orden de los vértices.
  *Fundamento: EUDR GeoJSON File Description v1.5*
- **[ING_MISSING_LATLON]** — Parcela `MSNS-003-MISSINGLATLON`: Falta Latitude o Longitude.
  *Fundamento: Producto 1 — esquema de entrada tabular*
- **[ING_TWO_VERTICES]** — Parcela `MSNS-004-TWOVERTICES`: La parcela tiene exactamente 2 filas de vértices: no alcanza para un polígono (mínimo 3 vértices distintos) y no es un único punto. Revisar si falta un vértice o si se declaró de más.
  *Fundamento: Producto 1 — esquema de entrada tabular*
- **[PARCELS_OVERLAP]** — Parcela `MSNS-010-OVERLAP-A / MSNS-010-OVERLAP-B`: La parcela 'MSNS-010-OVERLAP-A' se superpone con 'MSNS-010-OVERLAP-B' en 7.170 ha.
  *Fundamento: EUDR GeoJSON File Description v1.5*
- **[RULE_4HA_VIOLATION]** — Parcela `MSNS-008-4HAVIOLATION`: Area declarada = 8.0 ha (> 4 ha) pero la geometría es un Point. Parcelas de más de 4 ha deben declararse como polígono, con vértices suficientes para describir el perímetro.
  *Fundamento: Reglamento (UE) 2023/1115, Art. 9*

## 🟡 Advertencias (6)

- **[AREA_MISMATCH]** — Parcela `MSNS-013-AREAMISMATCH`: El área calculada de la geometría (14.94 ha) supera 4 ha pero el Area declarada (1.5 ha) no. Revisar cuál de los dos valores es correcto.
  *Fundamento: EUDR GeoJSON File Description v1.5*
- **[COORD_PRECISION]** — Parcela `MSNS-007-PRECISION`: La longitud tiene 14 decimales (máximo recomendado: 6). Más de 6 decimales puede generar coordenadas duplicadas por redondeo al procesarse en el sistema.
  *Fundamento: EUDR GeoJSON File Description v1.5*
- **[COORD_PRECISION]** — Parcela `MSNS-007-PRECISION`: La latitud tiene 14 decimales (máximo recomendado: 6). Más de 6 decimales puede generar coordenadas duplicadas por redondeo al procesarse en el sistema.
  *Fundamento: EUDR GeoJSON File Description v1.5*
- **[ING_METADATA_INCONSISTENT]** — Parcela `MSNS-005-INCONSISTENT`: La propiedad 'ProducerCountry' tiene valores distintos entre filas de la misma parcela (['AR', 'BR']); se usó el primero.
  *Fundamento: Producto 1 — consistencia de metadatos por ParcelID*
- **[POINT_NO_AREA]** — Parcela `MSNS-009-NOAREA`: La parcela es un Point sin Area declarada. El Information System asumirá 4 ha por defecto — riesgo silencioso si la parcela real es más grande o más chica.
  *Fundamento: EUDR GeoJSON File Description v1.5*
- **[SPECIES_MISSING]** — Parcela `MSNS-012-NOSPECIES`: Falta(n) CommonName, ScientificName. Para madera, la DDS exige nombre común y nombre científico completo de la especie.
  *Fundamento: Reglamento de Ejecución (UE) 2024/3084, Art. 4*

## ℹ️ Nota de conversións (6)

- **[ING_AUTOCLOSE]** — Parcela `MSNS-002`: El polígono se cerró automáticamente repitiendo el primer vértice al final (el archivo de origen no lo traía repetido, práctica habitual en planillas).
  *Fundamento: Producto 1 — conversión tabular → GeoJSON*
- **[ING_AUTOCLOSE]** — Parcela `MSNS-005-INCONSISTENT`: El polígono se cerró automáticamente repitiendo el primer vértice al final (el archivo de origen no lo traía repetido, práctica habitual en planillas).
  *Fundamento: Producto 1 — conversión tabular → GeoJSON*
- **[ING_AUTOCLOSE]** — Parcela `MSNS-010-OVERLAP-A`: El polígono se cerró automáticamente repitiendo el primer vértice al final (el archivo de origen no lo traía repetido, práctica habitual en planillas).
  *Fundamento: Producto 1 — conversión tabular → GeoJSON*
- **[ING_AUTOCLOSE]** — Parcela `MSNS-010-OVERLAP-B`: El polígono se cerró automáticamente repitiendo el primer vértice al final (el archivo de origen no lo traía repetido, práctica habitual en planillas).
  *Fundamento: Producto 1 — conversión tabular → GeoJSON*
- **[ING_AUTOCLOSE]** — Parcela `MSNS-013-AREAMISMATCH`: El polígono se cerró automáticamente repitiendo el primer vértice al final (el archivo de origen no lo traía repetido, práctica habitual en planillas).
  *Fundamento: Producto 1 — conversión tabular → GeoJSON*
- **[ING_AUTOCLOSE]** — Parcela `MSNS-014-SELFINTERSECT`: El polígono se cerró automáticamente repitiendo el primer vértice al final (el archivo de origen no lo traía repetido, práctica habitual en planillas).
  *Fundamento: Producto 1 — conversión tabular → GeoJSON*


## 6. Repetir con el archivo GeoJSON crudo

Ejercita el otro camino de ingesta: un proveedor que ya manda GeoJSON, pero con errores
estructurales (polígonos sin cerrar, tipos de geometría prohibidos, capitalización de
propiedades incorrecta, etc.).

In [7]:
result_geojson = run("../data/synthetic_geojson_input.geojson", "../outputs/demo_geojson")

print(f"Parcelas en el GeoJSON de salida: {result_geojson['n_output_features']}")
print(f"Parcelas excluidas por error bloqueante: {result_geojson['n_excluded_parcels']}")
print(f"Hallazgos totales: {len(result_geojson['findings'])}")

Parcelas en el GeoJSON de salida: 7
Parcelas excluidas por error bloqueante: 15
Hallazgos totales: 49


## 7. Set de prueba: verificación automática

`tests/run_validation_test.py` corre esto mismo contra los 37 errores sembrados a propósito y
verifica, código por código, que el validador los detecta todos. Se puede correr desde acá:

In [8]:
!cd .. && python3 tests/run_validation_test.py


TEST 1 — GeoJSON crudo (synthetic_geojson_input.geojson)


  ✅ CTES-001: sin hallazgos (control válido)
  ✅ CTES-002: sin hallazgos (control válido)
  ✅ CTES-003-LINESTRING: OK ['GEOM_TYPE_PROHIBITED']
  ✅ CTES-004-GEOMCOLLECTION: OK ['GEOM_TYPE_PROHIBITED']
  ✅ CTES-005-HOLE: OK ['POLY_HAS_HOLES']  [+ extra no evaluados: ['SPECIES_MISSING']]
  ✅ CTES-006-UNCLOSED: OK ['POLY_NOT_CLOSED']  [+ extra no evaluados: ['SPECIES_MISSING']]
  ✅ CTES-007-SELFINTERSECT: OK ['GEOM_SELF_INTERSECTION']  [+ extra no evaluados: ['COORD_PRECISION', 'SPECIES_MISSING']]
  ✅ CTES-008-PROPCASE: OK ['PROPERTY_CASE']  [+ extra no evaluados: ['SPECIES_MISSING']]
  ✅ CTES-009-PRECISION: OK ['COORD_PRECISION']  [+ extra no evaluados: ['SPECIES_MISSING']]
  ✅ CTES-010-COUNTRYMISMATCH: OK ['COUNTRY_MISMATCH']  [+ extra no evaluados: ['SPECIES_MISSING']]
  ✅ CTES-011-INVALIDISO: OK ['COUNTRY_INVALID_ISO']  [+ extra no evaluados: ['SPECIES_MISSING']]
  ✅ CTES-012-COUNTRYMISSING: OK ['COUNTRY_MISSING']  [+ extra no evaluados: ['SPECIES_MISSING']]
  ✅ CTES-013-AREASTRING: OK

  ✅ MSNS-001: sin hallazgos (control válido)
  ✅ MSNS-002: OK ['ING_AUTOCLOSE']
  ✅ MSNS-003-MISSINGLATLON: OK ['ING_MISSING_LATLON']
  ✅ MSNS-004-TWOVERTICES: OK ['ING_TWO_VERTICES']
  ✅ MSNS-005-INCONSISTENT: OK ['ING_METADATA_INCONSISTENT']  [+ extra no evaluados: ['ING_AUTOCLOSE']]
  ✅ MSNS-006-LATLONSWAP: OK ['COUNTRY_MISMATCH']
  ✅ MSNS-007-PRECISION: OK ['COORD_PRECISION']
  ✅ MSNS-008-4HAVIOLATION: OK ['RULE_4HA_VIOLATION']
  ✅ MSNS-009-NOAREA: OK ['POINT_NO_AREA']
  ✅ MSNS-010-OVERLAP-A: OK ['PARCELS_OVERLAP']  [+ extra no evaluados: ['ING_AUTOCLOSE']]
  ✅ MSNS-010-OVERLAP-B: OK ['PARCELS_OVERLAP']  [+ extra no evaluados: ['ING_AUTOCLOSE']]
  ✅ MSNS-011-INVALIDISO: OK ['COUNTRY_INVALID_ISO']
  ✅ MSNS-012-NOSPECIES: OK ['SPECIES_MISSING']
  ✅ MSNS-013-AREAMISMATCH: OK ['AREA_MISMATCH']  [+ extra no evaluados: ['ING_AUTOCLOSE']]
  ✅ MSNS-014-SELFINTERSECT: OK ['GEOM_SELF_INTERSECTION']  [+ extra no evaluados: ['ING_AUTOCLOSE']]

  Total hallazgos: 19
  Parcelas en GeoJSON de sal